In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import time
import json

# Como estamos en Google Colab, podemos permitirnos generar embeddigs de más frases (en este caso hacemos 20.000)
# Después se puede limitar la cantidad cuando vayamos a insertar.
# Los resultados se guardan en embeddings.json para poder usarlos posteriormente en el resto de scripts.

# carga el dataset, que tiene una única columna con la frases
# https://huggingface.co/docs/datasets/en/loading
dataset = load_dataset("SamuelYang/bookcorpus", split="train[:20000]")
sentences = dataset['text']

# carga del transformer que usaremos para generar los embeddings de cada frase
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#usage-sentence-transformers
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# genera los embeddings
start = time.perf_counter()
embeddings = model.encode(sentences)
end = time.perf_counter()

# metadatos
count = embeddings.shape[0]
dimensions = embeddings.shape[1]
elapsed = end - start

assert count == len(embeddings)
assert count == len(sentences)
assert dimensions == 384 # de momento, el tamaño del vector está hardcodeado en nuestra implementación

rows = []
for id in range(count):
  rows.append({
    "id": id,
    "sentence": sentences[id],
    "embedding": embeddings[id].tolist()
  })

result = {
  "elapsed": elapsed,
  "count": count,
  "dimensions": dimensions,
  "rows": rows
}

with open('embeddings.json', 'w') as f:
  json.dump(result, f)
